<a href="https://colab.research.google.com/github/shobhittya/Quantum_Image_Forensics/blob/main/qfusion_pipeline_colab_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantum fusion: Full experiment pipeline
**Order:** 1 setup, 2 write files, 3 synthetic dry run, 4 real data (**Option A needs no dataset access**; Option B needs FaceForensics++).

Use a **GPU runtime** (Runtime > Change runtime type) for step 4. Step 3 works on CPU.

## 1. Setup

In [2]:
# OpenCV is only needed for the VIDEO track (4B). The image track (4A) does not use it, so a broken cv2 must not block you.
!pip install -q tabulate datasets
try:
    import cv2
    print('cv2 OK' if hasattr(cv2, 'CascadeClassifier') else 'WARNING: cv2 is broken - fine for the image track (4A); fix it before the video track (4B) with the repair cell + Runtime > Restart session')
except Exception as e:
    print('WARNING: cv2 import failed:', e, '- fine for the image track (4A)')
from google.colab import drive
drive.mount('/content/drive')   # keeps frames/features/results if the session disconnects

cv2 OK
Mounted at /content/drive


In [3]:
# REPAIR (only if the cell above failed): remove all OpenCV builds, install one, then Runtime > Restart session and re-run from the top
!pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python
!pip install -q opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 12.7 MB/s eta 0:00:00


## 2. Write the code files

In [4]:
%%writefile qfusion.py
"""Hybrid quantum-classical spatial/frequency fusion (draft v0.2, Sections 4.3-4.9).
Pure-PyTorch batched statevector simulator -> no quantum SDK needed for ideal simulation.
Needs: torch, numpy. NOTE: written against the draft; run `python smoke_test.py` first."""
import math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F

# ---------- statevector helpers (qubit 0 = most significant bit) ----------
def _bit(j, q, n): return (j >> (n - 1 - q)) & 1

def entangler_perm(n, ring=False):
    """Basis-index permutation implementing Ex (Eq. 6): CNOT s_i->f_i, plus f_i->s_{i+1 mod m} if ring."""
    m = n // 2
    pairs = [(i, m + i) for i in range(m)]
    if ring: pairs += [(m + i, (i + 1) % m) for i in range(m)]
    j = np.arange(2 ** n); f = j.copy()
    for c, t in pairs: f = f ^ (_bit(f, c, n) << (n - 1 - t))
    inv = np.empty_like(f); inv[f] = j            # new[f(j)] = old[j]  ->  new = old[inv]
    return torch.as_tensor(inv, dtype=torch.long)

def z_signs(n):
    j = np.arange(2 ** n)
    return torch.as_tensor(np.stack([1 - 2 * _bit(j, k, n) for k in range(n)], 1), dtype=torch.float32)

def angle_encode(z):
    """Eq. (1): prod_i RY(z_i)|0>.  z: (B,n) -> real statevector (B,2^n)."""
    B, n = z.shape
    psi = torch.ones(B, 1, device=z.device, dtype=z.dtype)
    for i in range(n):
        v = torch.stack([torch.cos(z[:, i] / 2), torch.sin(z[:, i] / 2)], 1)   # (B,2)
        psi = (psi[:, :, None] * v[:, None, :]).reshape(B, -1)
    return psi

def apply_1q(psi, U, q, n):
    """Apply a fixed 2x2 gate U (complex) to qubit q of psi (B,2^n)."""
    B = psi.shape[0]; v = psi.reshape(B, 2 ** q, 2, 2 ** (n - q - 1))
    return torch.einsum('ij,bajc->baic', U.to(v.dtype), v).reshape(B, -1)

def apply_ry_batch(psi, ang, q, n):
    """Per-sample RY(ang[b]) on qubit q (real states; used for the kernel feature map)."""
    B = psi.shape[0]; v = psi.reshape(B, 2 ** q, 2, 2 ** (n - q - 1))
    c = torch.cos(ang / 2).view(B, 1, 1); s = torch.sin(ang / 2).view(B, 1, 1)
    a, b = v[:, :, 0], v[:, :, 1]
    return torch.stack([c * a - s * b, s * a + c * b], 2).reshape(B, -1)

def ry_mat(t): c, s = torch.cos(t / 2), torch.sin(t / 2); return torch.stack([torch.stack([c, -s]), torch.stack([s, c])]).to(torch.complex64)
def rz_mat(t):
    e = torch.exp(-0.5j * t.to(torch.complex64)); z = torch.zeros_like(e)
    return torch.stack([torch.stack([e, z]), torch.stack([z, e.conj()])])

# ---------- fusion blocks ----------
class QuantumFusion(nn.Module):
    """Variational entangling fusion, Eqs. (5)-(7). Output: n Pauli-Z expectations."""
    def __init__(self, n=8, layers=2, ring=False, cross_entangle=True):
        super().__init__()
        self.n, self.L = n, layers
        th = torch.empty(layers, n, 2)                                           # 2nL params
        th[..., 0].normal_(math.pi / 2, 0.3)                                     # RY offsets: readout ~ -sin(z), odd in z (a near-zero init gives cos(z), blind to linear signal)
        th[..., 1].uniform_(0, 2 * math.pi)                                      # RZ angles
        self.theta = nn.Parameter(th)
        self.register_buffer('perm', entangler_perm(n, ring))
        self.register_buffer('zs', z_signs(n))
        self.cross = cross_entangle                                              # False = ablation
    def forward(self, z):
        psi = angle_encode(z).to(torch.complex64)
        for l in range(self.L):
            if self.cross: psi = psi[:, self.perm]
            for q in range(self.n):
                psi = apply_1q(psi, rz_mat(self.theta[l, q, 1]) @ ry_mat(self.theta[l, q, 0]), q, self.n)
        return (psi.abs() ** 2) @ self.zs

def kernel_state(z, perm, use_E=True):
    """Feature map |psi(z)> = S(z) Ex S(z)|0>  (Section 4.7, re-uploading)."""
    n = z.shape[1]; psi = angle_encode(z)
    if use_E: psi = psi[:, perm]
    for q in range(n): psi = apply_ry_batch(psi, z[:, q], q, n)
    return psi

def fidelity_gram(za, zb, perm, use_E=True):
    A, Bm = kernel_state(za, perm, use_E), kernel_state(zb, perm, use_E)
    return (A @ Bm.T) ** 2                                                       # real states: |<a|b>|^2

class TrigProductHead(nn.Module):
    """Classical baseline matching the paired circuit's function class (Section 4.9)."""
    def __init__(self, n=8):
        super().__init__(); m = n // 2; self.m = m
        self.lin = nn.Linear(9 * m, n)
    def forward(self, z):
        m = self.m; s, f = z[:, :m], z[:, m:]
        vs = torch.stack([torch.ones_like(s), torch.sin(s), torch.cos(s)], -1)   # (B,m,3)
        vf = torch.stack([torch.ones_like(f), torch.sin(f), torch.cos(f)], -1)
        return self.lin((vs[..., :, None] * vf[..., None, :]).flatten(1))

class ConcatMLP(nn.Module):
    def __init__(self, n=8, hidden=16): super().__init__(); self.net = nn.Sequential(nn.Linear(n, hidden), nn.Tanh(), nn.Linear(hidden, n))
    def forward(self, z): return self.net(z)

# ---------- streams and full model ----------
def small_cnn(out_dim, in_ch=3):
    return nn.Sequential(nn.Conv2d(in_ch, 16, 3, 2, 1), nn.ReLU(), nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, out_dim))

class FusionDetector(nn.Module):
    """x: (B,3,H,W) frame batch -> logit.  fusion in {'quantum','quantum_ring','quantum_noent','trig','concat'}."""
    def __init__(self, fusion='quantum', n_s=4, n_f=4, layers=2, feat=64):
        super().__init__()
        self.ns, self.nf = n_s, n_f; n = n_s + n_f
        self.spatial, self.freq = small_cnn(feat), small_cnn(feat)               # replace with real backbones (Sec. 4.3)
        self.ps, self.pf = nn.Linear(feat, n_s), nn.Linear(feat, n_f)
        self.fusion_name = fusion
        self.fuse = {'quantum': lambda: QuantumFusion(n, layers),
                     'quantum_ring': lambda: QuantumFusion(n, layers, ring=True),
                     'quantum_noent': lambda: QuantumFusion(n, layers, cross_entangle=False),
                     'trig': lambda: TrigProductHead(n), 'concat': lambda: ConcatMLP(n)}[fusion]()
        self.cls = nn.Linear(n, 1)
    @staticmethod
    def freq_input(x):                                                          # log-magnitude FFT (Sec. 4.4)
        return torch.log1p(torch.fft.fftshift(torch.fft.fft2(x.mean(1, keepdim=True)), dim=(-2, -1)).abs()).expand(-1, 3, -1, -1)
    def forward(self, x):
        zs = math.pi * torch.tanh(self.ps(self.spatial(x)))                      # Eqs. (3)-(4)
        zf = math.pi * torch.tanh(self.pf(self.freq(self.freq_input(x))))
        return self.cls(self.fuse(torch.cat([zs, zf], 1))).squeeze(1)            # train with BCE-with-logits (Eq. 9)

def video_score(model, clip):                                                    # Sec. 4.8: average-pool frame scores
    return torch.sigmoid(model(clip)).mean()


Writing qfusion.py


In [5]:
%%writefile splits.py
"""Leakage-free splits: videos that share an identity (e.g. FF++ '123_456' and '456_123' and real '123', '456')
end up in the same connected component, and whole components go to one split."""
import re, random

def identity_ids(stem, mode):
    if mode == 'ffpp':    return re.findall(r'\d+', stem)[:2]      # '123' or '123_456'
    if mode == 'celebdf': return re.findall(r'id(\d+)', stem)      # 'id0_id1_0003', 'id0_0001'
    return []                                                      # 'stem': every video is its own group

def assign_splits(video_ids, ratios=(0.7, 0.15, 0.15), seed=0):
    """video_ids: {video_key: [identity tokens]} -> {video_key: 'train'|'val'|'test'}"""
    parent = {}
    def find(a):
        parent.setdefault(a, a)
        while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for v, ids in video_ids.items():
        toks = ['id:' + i for i in ids] or ['vid:' + v]
        for t in toks[1:]: parent[find(t)] = find(toks[0])
        find(toks[0])
    comps = {}
    for v, ids in video_ids.items():
        comps.setdefault(find(('id:' + ids[0]) if ids else 'vid:' + v), []).append(v)
    groups = list(comps.values()); random.Random(seed).shuffle(groups)
    names = ['train', 'val', 'test']; N = len(video_ids)
    target = [r * N for r in ratios]; count = [0, 0, 0]; out = {}
    for g in groups:
        k = max(range(3), key=lambda i: target[i] - count[i])
        count[k] += len(g)
        for v in g: out[v] = names[k]
    return out


Writing splits.py


In [6]:
%%writefile metrics.py
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

def eer(y, s):
    fpr, tpr, _ = roc_curve(y, s); fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr))); return float((fpr[i] + fnr[i]) / 2)

def _auc(y, s):
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else float('nan')

def evaluate(prob, label, video):
    """Frame-level and video-level (mean frame probability per video) AUC / EER / accuracy."""
    prob, label = np.asarray(prob, float), np.asarray(label, int)
    out = {'frame_auc': _auc(label, prob), 'frame_acc': float(((prob > 0.5) == label).mean())}
    out['frame_eer'] = eer(label, prob) if len(np.unique(label)) == 2 else float('nan')
    df = pd.DataFrame({'v': video, 'p': prob, 'y': label}).groupby('v').agg(p=('p', 'mean'), y=('y', 'first'))
    out['video_auc'] = _auc(df.y.values, df.p.values)
    out['video_eer'] = eer(df.y.values, df.p.values) if df.y.nunique() == 2 else float('nan')
    out['video_acc'] = float(((df.p.values > 0.5) == df.y.values).mean())
    return out


Writing metrics.py


In [7]:
%%writefile prepare_hf_images.py
"""Balanced subset of a Hugging Face real/fake image dataset -> PNG files + manifest_<name>.csv
No account needed.  Default: 140k Real and Fake Faces (FFHQ real vs StyleGAN fake):
  python prepare_hf_images.py --repo TheKernel01/140k-Real-and-Fake-Faces --name faces140k --train 6000 --val 1000 --test 2000"""
import argparse, csv, os, numpy as np
from datasets import load_dataset
from splits import assign_splits
ap = argparse.ArgumentParser(); ap.add_argument('--repo', default='TheKernel01/140k-Real-and-Fake-Faces'); ap.add_argument('--name', default='faces140k')
ap.add_argument('--out', default='images'); ap.add_argument('--train', type=int, default=6000); ap.add_argument('--val', type=int, default=1000)
ap.add_argument('--test', type=int, default=2000); ap.add_argument('--seed', type=int, default=0)
a = ap.parse_args(); rng = np.random.default_rng(a.seed)
want = {'train': a.train, 'val': a.val, 'test': a.test}; alias = {'train': 'train', 'validation': 'val', 'valid': 'val', 'val': 'val', 'test': 'test'}
dd = load_dataset(a.repo); print('splits found:', {k: len(v) for k, v in dd.items()}); rows = []
for sp_name, d in dd.items():
    sp = alias.get(sp_name)
    if sp is None: continue
    names = d.features['label'].names if hasattr(d.features['label'], 'names') else ['real', 'fake']
    fake_ids = {i for i, nm in enumerate(names) if nm.lower() in ('fake', '1')} or {1}
    print(sp_name, '-> split', sp, '| label names', names, '| fake ids', fake_ids)       # check this mapping once
    lab = np.array([1 if int(l) in fake_ids else 0 for l in d['label']]); pick = []
    for c in (0, 1):
        idx = np.where(lab == c)[0]; pick += list(rng.choice(idx, min(want[sp] // 2, len(idx)), replace=False))
    for i in sorted(pick):
        p = f'{a.out}/{a.name}/{sp}/{lab[i]}'; os.makedirs(p, exist_ok=True); fn = f'{p}/{i}.png'
        if not os.path.exists(fn): d[int(i)]['image'].convert('RGB').save(fn)
        rows.append((fn, int(lab[i]), a.name, f'{a.name}_{sp}_{i}', sp))
    print(sp, 'saved', len(pick))
if not {r[4] for r in rows} >= {'train', 'val', 'test'}:                                 # dataset without its own splits
    sp = assign_splits({r[3]: [] for r in rows}, seed=a.seed); rows = [(f, y, n, v, sp[v]) for f, y, n, v, _ in rows]
with open(f'manifest_{a.name}.csv', 'w', newline='') as fh:
    w = csv.writer(fh); w.writerow(['path', 'label', 'dataset', 'video', 'split']); w.writerows(rows)
print('manifest rows:', len(rows))


Writing prepare_hf_images.py


In [8]:
%%writefile prepare_images.py
"""Folders of images -> manifest_<dataset>.csv (no copying; leakage-free random split per image).
Use for any real/fake image set (e.g. a diffusion-generated set for the cross-generator test).
  python prepare_images.py --dataset diffusion --real_dir /path/real --fake_dir /path/fake1 /path/fake2 --max_per_class 3000"""
import argparse, csv, glob, os, random
from splits import assign_splits
ap = argparse.ArgumentParser(); ap.add_argument('--dataset', required=True)
ap.add_argument('--real_dir', nargs='+', required=True); ap.add_argument('--fake_dir', nargs='+', required=True)
ap.add_argument('--max_per_class', type=int, default=5000); ap.add_argument('--seed', type=int, default=0)
a = ap.parse_args(); EXT = ('.jpg', '.jpeg', '.png', '.webp'); rng = random.Random(a.seed); rows = {}
for label, dirs in ((0, a.real_dir), (1, a.fake_dir)):
    files = sorted(p for d in dirs for p in glob.glob(os.path.join(d, '**', '*'), recursive=True) if p.lower().endswith(EXT))
    rng.shuffle(files); files = files[:a.max_per_class]; print('real' if label == 0 else 'fake', len(files), 'images')
    for i, p in enumerate(files): rows[f'{a.dataset}_{label}_{i}'] = (p, label)
if not rows: raise SystemExit('No images found in the given folders - check the paths. Nothing written.')
split = assign_splits({k: [] for k in rows}, seed=a.seed)
with open(f'manifest_{a.dataset}.csv', 'w', newline='') as fh:
    w = csv.writer(fh); w.writerow(['path', 'label', 'dataset', 'video', 'split'])
    for k, (p, y) in rows.items(): w.writerow([p, y, a.dataset, k, split[k]])
print('manifest rows:', len(rows))


Writing prepare_images.py


In [9]:
%%writefile extract_frames.py
"""Videos -> face-cropped JPEG frames + manifest_<dataset>.csv (with leakage-free splits).
Example (FF++ c23):
  python extract_frames.py --dataset ffpp --id_mode ffpp --out frames \
     --real FaceForensics/original_sequences/youtube/c23/videos \
     --fake FaceForensics/manipulated_sequences/{Deepfakes,Face2Face,FaceSwap,NeuralTextures}/c23/videos --crop_face"""
import argparse, csv, glob, hashlib, os
import cv2, numpy as np
from splits import identity_ids, assign_splits

ap = argparse.ArgumentParser()
ap.add_argument('--dataset', required=True); ap.add_argument('--out', default='frames')
ap.add_argument('--real', nargs='+', required=True); ap.add_argument('--fake', nargs='+', required=True)
ap.add_argument('--id_mode', default='stem', choices=['ffpp', 'celebdf', 'stem'])
ap.add_argument('--frames_real', type=int, default=32); ap.add_argument('--frames_fake', type=int, default=8)
ap.add_argument('--size', type=int, default=224); ap.add_argument('--crop_face', action='store_true')
ap.add_argument('--max_videos', type=int, default=0, help='debug: cap videos per class')
ap.add_argument('--seed', type=int, default=0)
a = ap.parse_args()
cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def crop(f):
    g = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
    faces = cascade.detectMultiScale(g, 1.1, 5, minSize=(64, 64))
    if len(faces) == 0: return None
    x, y, w, h = max(faces, key=lambda r: r[2] * r[3]); s = int(max(w, h) * 1.3 / 2); cx, cy = x + w // 2, y + h // 2
    H, W = f.shape[:2]; return f[max(cy - s, 0):min(cy + s, H), max(cx - s, 0):min(cx + s, W)]

def grab(path, k):
    cap = cv2.VideoCapture(path); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); out = []
    for i in np.linspace(0, max(n - 1, 0), k + 2)[1:-1].astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, f = cap.read()
        if ok: out.append(f)
    cap.release(); return out

vids = []
for label, dirs, k in (('real', a.real, a.frames_real), ('fake', a.fake, a.frames_fake)):
    found = sorted(p for d in dirs for p in glob.glob(os.path.join(d, '**', '*.mp4'), recursive=True))
    if a.max_videos: found = found[:a.max_videos]
    print(label, len(found), 'videos'); vids += [(p, label, k) for p in found]

rows, ids_of = [], {}
for n, (p, label, k) in enumerate(vids):
    stem = os.path.splitext(os.path.basename(p))[0]
    key = f"{stem}_{hashlib.md5(p.encode()).hexdigest()[:6]}"
    d = os.path.join(a.out, a.dataset, label, key); os.makedirs(d, exist_ok=True)
    files = sorted(glob.glob(os.path.join(d, '*.jpg')))
    if not files:                                                  # resume-friendly
        for j, f in enumerate(grab(p, k)):
            if a.crop_face:
                f = crop(f)
                if f is None: continue
            fn = os.path.join(d, f'f{j:03d}.jpg')
            cv2.imwrite(fn, cv2.resize(f, (a.size, a.size)), [cv2.IMWRITE_JPEG_QUALITY, 95]); files.append(fn)
    if files:
        ids_of[key] = identity_ids(stem, a.id_mode)
        rows += [(fn, 1 if label == 'fake' else 0, a.dataset, key) for fn in files]
    if n % 50 == 0: print(f'{n}/{len(vids)}', flush=True)

split = assign_splits(ids_of, seed=a.seed)
with open(f'manifest_{a.dataset}.csv', 'w', newline='') as fh:
    w = csv.writer(fh); w.writerow(['path', 'label', 'dataset', 'video', 'split'])
    for fn, y, ds, key in rows: w.writerow([fn, y, ds, key, split[key]])
print('frames:', len(rows), 'videos:', len(ids_of), 'dropped (no face/frames):', len(vids) - len(ids_of))


Writing extract_frames.py


In [10]:
%%writefile cache_features.py
"""Frozen ImageNet ResNet-18 features for both streams -> features_<dataset>.pt
spatial stream: RGB frame.  frequency stream: log-magnitude 2-D FFT of the grayscale frame (replicated to 3 channels)."""
import argparse, csv, numpy as np, torch, torch.nn as nn, torchvision
from PIL import Image

ap = argparse.ArgumentParser(); ap.add_argument('--manifest', required=True); ap.add_argument('--out', default='.')
ap.add_argument('--bs', type=int, default=128); ap.add_argument('--workers', type=int, default=2)
a = ap.parse_args()
rows = list(csv.DictReader(open(a.manifest))); ds_name = rows[0]['dataset']
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

class DS(torch.utils.data.Dataset):
    def __len__(s): return len(rows)
    def __getitem__(s, i):
        im = np.asarray(Image.open(rows[i]['path']).convert('RGB').resize((224, 224)))
        return torch.from_numpy(im.copy()).permute(2, 0, 1).float() / 255, i

def backbone():
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Identity(); return m.eval().to(dev).requires_grad_(False)
net_s, net_f = backbone(), backbone()                         # separate frozen copies for the two streams
mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

def freq_input(x):
    m = torch.log1p(torch.fft.fftshift(torch.fft.fft2(x.mean(1)), dim=(-2, -1)).abs())
    m = (m - m.mean((1, 2), keepdim=True)) / (m.std((1, 2), keepdim=True) + 1e-6)
    return m.unsqueeze(1).expand(-1, 3, -1, -1)

S, Fq = torch.zeros(len(rows), 512), torch.zeros(len(rows), 512)
with torch.no_grad():
    for n, (x, idx) in enumerate(torch.utils.data.DataLoader(DS(), batch_size=a.bs, num_workers=a.workers)):
        x = x.to(dev); S[idx] = net_s((x - mean) / std).cpu(); Fq[idx] = net_f(freq_input(x)).cpu()
        if n % 20 == 0: print(n * a.bs, '/', len(rows), flush=True)
torch.save({'spatial': S, 'freq': Fq, 'label': torch.tensor([int(r['label']) for r in rows]),
            'video': [r['video'] for r in rows], 'split': [r['split'] for r in rows], 'dataset': ds_name,
            'path': [r['path'] for r in rows]}, f'{a.out}/features_{ds_name}.pt')
print('saved', f'{a.out}/features_{ds_name}.pt')


Writing cache_features.py


In [11]:
%%writefile make_synthetic.py
"""Synthetic cached features so train_eval.py can be tested end-to-end without any dataset.
synth_a: training dataset.  synth_b: same signal with a domain shift (acts as the cross-dataset set)."""
import numpy as np, torch
from splits import assign_splits
rng = np.random.default_rng(0)
mu_s, mu_f, alt_s, alt_f = (rng.normal(size=512) for _ in range(4))
for name, shift in (('synth_a', 0.0), ('synth_b', 0.5)):
    V, K = 300, 8; y_v = rng.integers(0, 2, V); lab, vid, xs, xf = [], [], [], []
    for v in range(V):
        off_s, off_f = rng.normal(size=512) * 0.5, rng.normal(size=512) * 0.5       # per-video nuisance
        for _ in range(K):
            ms = (1 - shift) * mu_s + shift * alt_s; mf = (1 - shift) * mu_f + shift * alt_f
            xs.append(rng.normal(size=512) + off_s + y_v[v] * 0.12 * ms); xf.append(rng.normal(size=512) + off_f + y_v[v] * 0.12 * mf)
            lab.append(y_v[v]); vid.append(f'{name}_v{v}')
    split = assign_splits({f'{name}_v{v}': [] for v in range(V)})
    torch.save({'spatial': torch.tensor(np.array(xs), dtype=torch.float32), 'freq': torch.tensor(np.array(xf), dtype=torch.float32),
                'label': torch.tensor(lab), 'video': vid, 'split': [split[v] for v in vid], 'dataset': name, 'path': [''] * len(lab)},
               f'features_{name}.pt')
    print('saved', f'features_{name}.pt', len(lab), 'frames')


Writing make_synthetic.py


In [12]:
%%writefile train_eval.py
"""Train/evaluate all fusion heads on cached features (Sections 5-6 of the draft).
  python train_eval.py --features_dir . --train_ds synth_a --cross_ds synth_b --seeds 2 --epochs 10
Outputs results/results.csv (per seed) and results/summary.csv + summary.md (mean +- std)."""
import argparse, copy, math, os, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.svm import SVC
from qfusion import QuantumFusion, TrigProductHead, fidelity_gram, entangler_perm
from metrics import evaluate

NN_HEADS = ['spatial_only', 'freq_only', 'concat', 'bilinear', 'crossattn', 'mlp_matched', 'trig', 'quantum', 'quantum_ring', 'quantum_noent']
KERNEL_HEADS = ['qkernel', 'qkernel_noent', 'rbf_svm']
ap = argparse.ArgumentParser()
ap.add_argument('--features_dir', default='.'); ap.add_argument('--train_ds', required=True); ap.add_argument('--cross_ds', nargs='*', default=[])
ap.add_argument('--heads', nargs='+', default=NN_HEADS); ap.add_argument('--kernel', action='store_true', help='also run quantum-kernel SVM variants')
ap.add_argument('--seeds', type=int, default=5); ap.add_argument('--epochs', type=int, default=30); ap.add_argument('--lr', type=float, default=3e-3)
ap.add_argument('--circ_lr', type=float, default=3e-2); ap.add_argument('--bs', type=int, default=256); ap.add_argument('--layers', type=int, default=2); ap.add_argument('--n_s', type=int, default=4); ap.add_argument('--n_f', type=int, default=4)
ap.add_argument('--kernel_n', type=int, default=2000); ap.add_argument('--kernel_eval_max', type=int, default=5000); ap.add_argument('--out', default='results')
a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; n = a.n_s + a.n_f

def load(name):
    d = torch.load(f'{a.features_dir}/features_{name}.pt', map_location='cpu', weights_only=False); d['video'] = np.array(d['video']); d['split'] = np.array(d['split']); return d
def subset(d, mask): return {k: (d[k][mask] if k in ('spatial', 'freq', 'label', 'video', 'split') else d[k]) for k in d}
D = load(a.train_ds); tr, va, te = (subset(D, D['split'] == s) for s in ('train', 'val', 'test'))
mu_s, sd_s = tr['spatial'].mean(0), tr['spatial'].std(0) + 1e-6; mu_f, sd_f = tr['freq'].mean(0), tr['freq'].std(0) + 1e-6
def prep(d):                                                    # standardize with TRAIN statistics only
    return ((d['spatial'] - mu_s) / sd_s).to(dev), ((d['freq'] - mu_f) / sd_f).to(dev), d['label'].float().to(dev)
eval_sets = {'in_test': te, **{f'cross_{c}': load(c) for c in a.cross_ds}}

class Net(nn.Module):
    def __init__(s, head):
        super().__init__(); s.head = head; s.ps, s.pf = nn.Linear(512, a.n_s), nn.Linear(512, a.n_f)    # trainable projections (Eqs. 3-4)
        if head == 'spatial_only': s.out = nn.Linear(a.n_s, 1)
        elif head == 'freq_only': s.out = nn.Linear(a.n_f, 1)
        elif head == 'concat': s.out = nn.Linear(n, 1)
        elif head == 'bilinear': s.out = nn.Linear(a.n_s * a.n_f, 1)
        elif head == 'crossattn':
            d = 8; s.es, s.ef = nn.Linear(a.n_s, d), nn.Linear(a.n_f, d); s.q, s.k, s.v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d); s.out = nn.Linear(d, 1)
        elif head == 'mlp_matched':                              # same #params as circuit(2nL)+classifier(n+1)
            h = max(1, round((2 * n * a.layers + n + 1 - 1) / (n + 2))); s.fuse = nn.Sequential(nn.Linear(n, h), nn.Tanh(), nn.Linear(h, 1))
        elif head == 'trig': s.fuse = TrigProductHead(n); s.out = nn.Linear(n, 1)
        elif head.startswith('quantum'):
            s.fuse = QuantumFusion(n, a.layers, ring=(head == 'quantum_ring'), cross_entangle=(head != 'quantum_noent')); s.out = nn.Linear(n, 1)
        else: raise ValueError(head)
    def n_head_params(s): return sum(p.numel() for k, p in s.named_parameters() if not k.startswith(('ps.', 'pf.')))
    def forward(s, xs, xf):
        zs, zf = math.pi * torch.tanh(s.ps(xs)), math.pi * torch.tanh(s.pf(xf)); h = s.head
        if h == 'spatial_only': return s.out(zs).squeeze(1)
        if h == 'freq_only': return s.out(zf).squeeze(1)
        if h == 'concat': return s.out(torch.cat([zs, zf], 1)).squeeze(1)
        if h == 'bilinear': return s.out((zs[:, :, None] * zf[:, None, :]).flatten(1)).squeeze(1)
        if h == 'crossattn':
            t = torch.stack([s.es(zs), s.ef(zf)], 1); att = torch.softmax(s.q(t) @ s.k(t).transpose(1, 2) / math.sqrt(t.shape[-1]), -1)
            return s.out((att @ s.v(t)).mean(1)).squeeze(1)
        z = torch.cat([zs, zf], 1)
        return (s.fuse(z).squeeze(1) if h == 'mlp_matched' else s.out(s.fuse(z)).squeeze(1))

@torch.no_grad()
def predict(net, Xs, Xf):
    net.eval(); return torch.cat([torch.sigmoid(net(Xs[i:i + 2048], Xf[i:i + 2048])) for i in range(0, len(Xs), 2048)]).cpu().numpy()

def fit(head, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    net = Net(head).to(dev)
    circ = [p for k, p in net.named_parameters() if k.endswith('theta')]            # circuit rotations: own LR, no weight decay
    rest = [p for k, p in net.named_parameters() if not k.endswith('theta')]
    groups = [{'params': rest, 'lr': a.lr, 'weight_decay': 1e-4}]
    if circ: groups.append({'params': circ, 'lr': a.circ_lr, 'weight_decay': 0.0})
    opt = torch.optim.Adam(groups)
    Xs, Xf, y = Xtr; pw = ((y == 0).sum() / (y == 1).sum().clamp(min=1)).to(dev); best = (-1, None)
    for ep in range(a.epochs):
        net.train(); perm = torch.randperm(len(y), device=dev)
        for i in range(0, len(y), a.bs):
            idx = perm[i:i + a.bs]; loss = F.binary_cross_entropy_with_logits(net(Xs[idx], Xf[idx]), y[idx], pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
        auc = evaluate(predict(net, *Xva[:2]), va['label'].numpy(), va['video'])['frame_auc']
        if auc > best[0]: best = (auc, copy.deepcopy(net.state_dict()))
    net.load_state_dict(best[1]); return net

# ---- quantum-kernel variant: PCA -> angles, fidelity Gram matrix of S(z) Ex S(z), precomputed-kernel SVM ----
def pca_fit(X, k):
    mu = X.mean(0); _, _, Vt = np.linalg.svd(X - mu, full_matrices=False); comp = Vt[:k]; return mu, comp, ((X - mu) @ comp.T).std(0) + 1e-6
def pca_angles(X, p): mu, comp, sd = p; return math.pi * np.clip(((X - mu) @ comp.T) / sd, -3, 3) / 3
def angles(d, ps_, pf_): return np.concatenate([pca_angles(d['spatial'].numpy(), ps_), pca_angles(d['freq'].numpy(), pf_)], 1).astype(np.float32)
def gram(A, B, use_E):
    perm = entangler_perm(n).to(dev); A, B = torch.tensor(A, device=dev), torch.tensor(B, device=dev)
    return torch.cat([fidelity_gram(A[i:i + 500], B, perm, use_E).cpu() for i in range(0, len(A), 500)]).numpy()

rows = []
Xtr, Xva = prep(tr), prep(va); Xev = {k: (prep(v), v) for k, v in eval_sets.items()}
for seed in range(a.seeds):
    for head in a.heads:
        net = fit(head, seed); npar = net.n_head_params()
        for name, ((Xs, Xf, _), d) in Xev.items():
            for k, v in evaluate(predict(net, Xs, Xf), d['label'].numpy(), d['video']).items(): rows.append(dict(seed=seed, head=head, eval_set=name, metric=k, value=v, head_params=npar))
        print(f'seed {seed} {head:14s} done', flush=True)
    if a.kernel:
        rng = np.random.default_rng(seed); y_all = tr['label'].numpy()
        sel = np.concatenate([rng.choice(np.where(y_all == c)[0], min(a.kernel_n // 2, (y_all == c).sum()), replace=False) for c in (0, 1)])
        ps_, pf_ = pca_fit(tr['spatial'].numpy(), a.n_s), pca_fit(tr['freq'].numpy(), a.n_f)
        Atr, ytr = angles(subset(tr, sel), ps_, pf_), y_all[sel]
        for head in KERNEL_HEADS:
            model = SVC(kernel='rbf', gamma='scale') if head == 'rbf_svm' else SVC(kernel='precomputed')
            model.fit(Atr if head == 'rbf_svm' else gram(Atr, Atr, head == 'qkernel'), ytr)
            for name, (_, d) in Xev.items():
                m = np.arange(len(d['label'])); m = rng.choice(m, a.kernel_eval_max, replace=False) if len(m) > a.kernel_eval_max else m
                dd = subset(d, m); Ae = angles(dd, ps_, pf_)
                s_ = model.decision_function(Ae if head == 'rbf_svm' else gram(Ae, Atr, head == 'qkernel'))
                for k, v in evaluate(1 / (1 + np.exp(-s_)), dd['label'].numpy(), dd['video']).items(): rows.append(dict(seed=seed, head=head, eval_set=name, metric=k, value=v, head_params=0))
            print(f'seed {seed} {head:14s} done', flush=True)
res = pd.DataFrame(rows); res.to_csv(f'{a.out}/results.csv', index=False)
g = res.groupby(['head', 'eval_set', 'metric']).value.agg(['mean', 'std']).reset_index()
g['txt'] = g['mean'].map('{:.3f}'.format) + ' ± ' + g['std'].fillna(0).map('{:.3f}'.format)
summ = g.pivot_table(index=['eval_set', 'head'], columns='metric', values='txt', aggfunc='first')[['frame_auc', 'frame_eer', 'video_auc', 'video_eer']]
summ.to_csv(f'{a.out}/summary.csv'); open(f'{a.out}/summary.md', 'w').write(summ.to_markdown()); print(summ.to_string())


Writing train_eval.py


## 3. Synthetic dry run (random features; checks the pipeline, not the science)

After the init/optimizer fix, `quantum` should land close to `concat` here (it was ~0.54 AUC before the fix).

In [13]:
!python make_synthetic.py
!python train_eval.py --train_ds synth_a --cross_ds synth_b --seeds 2 --epochs 8 --kernel --kernel_n 400

saved features_synth_a.pt 2400 frames
saved features_synth_b.pt 2400 frames
seed 0 spatial_only   done
seed 0 freq_only      done
seed 0 concat         done
seed 0 bilinear       done
seed 0 crossattn      done
seed 0 mlp_matched    done
seed 0 trig           done
seed 0 quantum        done
seed 0 quantum_ring   done
seed 0 quantum_noent  done
seed 0 qkernel        done
seed 0 qkernel_noent  done
seed 0 rbf_svm        done
seed 1 spatial_only   done
seed 1 freq_only      done
seed 1 concat         done
seed 1 bilinear       done
seed 1 crossattn      done
seed 1 mlp_matched    done
seed 1 trig           done
seed 1 quantum        done
seed 1 quantum_ring   done
seed 1 quantum_noent  done
seed 1 qkernel        done
seed 1 qkernel_noent  done
seed 1 rbf_svm        done
metric                           frame_auc      frame_eer      video_auc      video_eer
eval_set      head                                                                     
cross_synth_b bilinear       0.511 ± 0.000  0.

In [14]:
# RE-RUN THIS CELL after re-running the two %%writefile cells: training runs the copies in WORK, not /content.
WORK = '/content/drive/MyDrive/qfusion_work'
import os, shutil; os.makedirs(WORK, exist_ok=True)
for f in ['qfusion.py', 'splits.py', 'metrics.py', 'prepare_hf_images.py', 'prepare_images.py', 'extract_frames.py', 'cache_features.py', 'make_synthetic.py', 'train_eval.py']: shutil.copy('/content/'+f, WORK)
os.chdir(WORK)

## 4A. Image track: no dataset access needed
**140k Real and Fake Faces** (real = FFHQ photos, fake = StyleGAN) from Hugging Face. This is the 'GAN-generated image set' of Table I. The default subset is 6000 train / 1000 val / 2000 test images (about 10 minutes); raise the numbers for the final run. Watch the printed label mapping once to confirm `fake ids` is the StyleGAN class.

In [15]:
!python prepare_hf_images.py --repo TheKernel01/140k-Real-and-Fake-Faces --name faces140k --train 6000 --val 1000 --test 2000

README.md: 100% 5.06k/5.06k [00:00<00:00, 1.36MB/s]

data/train-00000-of-00006.parquet: downloading bytes:  66% 318M/478M [00:02<00:00, 247MB/s, 26.1MB/s  ]
data/train-00000-of-00006.parquet: downloading bytes:  80% 382M/478M [00:02<00:00, 259MB/s, 31.1MB/s  ]
data/train-00000-of-00006.parquet: downloading bytes:  96% 461M/478M [00:02<00:00, 250MB/s, 40.2MB/s  ]
data/train-00000-of-00006.parquet: reconstructing file:  84% 402M/478M [00:02<00:00, 181MB/s, 24.9MB/s  ]
data/train-00000-of-00006.parquet: downloading bytes: 100% 475M/475M [00:02<00:00, 161MB/s, 41.3MB/s  ]
data/train-00000-of-00006.parquet: reconstructing file: 100% 478M/478M [00:02<00:00, 162MB/s, 43.1MB/s  ]

data/train-00001-of-00006.parquet: downloading bytes:  84% 401M/479M [00:01<00:00, 485MB/s, 32.2MB/s  ]
data/train-00001-of-00006.parquet: downloading bytes:  99% 476M/479M [00:01<00:00, 430MB/s, 42.4MB/s  ]
data/train-00001-of-00006.parquet: downloading bytes: 100% 476M/476M [00:02<00:00, 179MB/s, 44.1MB/s  ]
data/t

In [19]:
!python cache_features.py --manifest manifest_faces140k.csv --out .

0 / 9000
2560 / 9000
5120 / 9000
7680 / 9000
saved ./features_faces140k.pt


In [17]:
!python train_eval.py --features_dir . --train_ds faces140k --seeds 5 --epochs 30 --kernel --out results_faces140k

seed 0 spatial_only   done
seed 0 freq_only      done
seed 0 concat         done
seed 0 bilinear       done
seed 0 crossattn      done
seed 0 mlp_matched    done
seed 0 trig           done
seed 0 quantum        done
seed 0 quantum_ring   done
seed 0 quantum_noent  done
seed 0 qkernel        done
seed 0 qkernel_noent  done
seed 0 rbf_svm        done
seed 1 spatial_only   done
seed 1 freq_only      done
seed 1 concat         done
seed 1 bilinear       done
seed 1 crossattn      done
seed 1 mlp_matched    done
seed 1 trig           done
seed 1 quantum        done
seed 1 quantum_ring   done
seed 1 quantum_noent  done
seed 1 qkernel        done
seed 1 qkernel_noent  done
seed 1 rbf_svm        done
seed 2 spatial_only   done
seed 2 freq_only      done
seed 2 concat         done
seed 2 bilinear       done
seed 2 crossattn      done
seed 2 mlp_matched    done
seed 2 trig           done
seed 2 quantum        done
seed 2 quantum_ring   done
seed 2 quantum_noent  done
seed 2 qkernel        done
s

### 4A-2. Cross-generator test (optional, needs a second image set)
Put any second real/fake image set in Drive (for example images from a diffusion model, with real photos beside them; GenImage subsets work) and point the folders below at it. Then add `--cross_ds diffusion` to the training command.

In [ ]:
# OPTIONAL: skip this cell unless you have a second real/fake image set. Set the two folders, then run.
import os
REAL = '/content/drive/MyDrive/diffusion_set/real'
FAKE = '/content/drive/MyDrive/diffusion_set/fake'
assert os.path.isdir(REAL) and os.path.isdir(FAKE), 'Skipping: no second image set found. This cell is optional.'
!python prepare_images.py --dataset diffusion --real_dir {REAL} --fake_dir {FAKE} --max_per_class 3000
!python cache_features.py --manifest manifest_diffusion.csv --out .
!python train_eval.py --features_dir . --train_ds faces140k --cross_ds diffusion --seeds 5 --epochs 30 --out results_cross_generator

## 4B. Video track (only when you have FaceForensics++)
FF++ is free for research but needs access: fill the Google form linked from the README at github.com/ondyari/FaceForensics; they email a download script. Standard folder layout assumed. Test with `--max_videos 20` first.

In [ ]:
FFPP = '/content/drive/MyDrive/FaceForensics'   # contains original_sequences/ and manipulated_sequences/
import os
assert os.path.isdir(FFPP), 'Skipping: FaceForensics++ not found. Section 4B is only for when you have FF++ access.'
!python extract_frames.py --dataset ffpp --id_mode ffpp --out frames --crop_face --frames_real 32 --frames_fake 8 \
  --real {FFPP}/original_sequences/youtube/c23/videos \
  --fake {FFPP}/manipulated_sequences/Deepfakes/c23/videos {FFPP}/manipulated_sequences/Face2Face/c23/videos {FFPP}/manipulated_sequences/FaceSwap/c23/videos {FFPP}/manipulated_sequences/NeuralTextures/c23/videos
!python cache_features.py --manifest manifest_ffpp.csv --out .

In [ ]:
# Celeb-DF as cross-dataset test (folders Celeb-real, YouTube-real, Celeb-synthesis)
CDF = '/content/drive/MyDrive/Celeb-DF-v2'
import os
assert os.path.isdir(CDF), 'Skipping: Celeb-DF not found. Only for the video track.'
!python extract_frames.py --dataset celebdf --id_mode celebdf --out frames --crop_face --frames_real 16 --frames_fake 8 \
  --real {CDF}/Celeb-real {CDF}/YouTube-real --fake {CDF}/Celeb-synthesis
!python cache_features.py --manifest manifest_celebdf.csv --out .
!python train_eval.py --features_dir . --train_ds ffpp --cross_ds celebdf --seeds 5 --epochs 30 --out results_ffpp

Results: `<out>/summary.md` (mean ± std AUC/EER), `results.csv` (every seed). Add `--kernel` to `train_eval.py` for the quantum-kernel SVM variants.